# 🥉 Bronze Layer — Raw Data Ingestion (JSON)

## Overview
The Bronze Layer ingests raw data **as-is** from source files and stores it in **one Delta table**.

Each row is **one JSON object**:
- `entity_type` — users, products, events, sales, or item_lookup
- `payload` — the full record as a JSON string
- `_ingested_at`, `_source_file` — pipeline metadata

Silver reads this single JSON bronze table and parses each entity back into typed tables.


In [1]:
import sys
from pathlib import Path

from pyspark.sql.functions import col, current_timestamp
import pyspark.sql.functions as F

# Import bronze JSON helpers from this folder
for candidate in [Path("."), Path("../Bronze")]:
    if (candidate / "bronze_json.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break

from bronze_json import BRONZE_JSON_PATH, to_bronze_json_rows

## Configuration — Paths & Sources

This cell defines all the paths used throughout the Bronze Layer notebook.
Rather than hardcoding paths in every cell, we centralize them here for 
maintainability — if a path changes, we only update it in one place.

### `RAW_PATH`
Points to the Unity Catalog Volume where the raw source files are stored.
This is a **read-only** zone — we never write to this location.

### `BRONZE_JSON_PATH`
Points to the single Bronze Delta table where every record is stored as one JSON object.
This is the **write destination** for this notebook.

### `PATHS`
A dictionary mapping each logical table name to its raw source location.
This allows us to reference sources by name (e.g. `PATHS["users"]`) 
rather than repeating the full path throughout the notebook.

In [2]:
RAW_PATH = "/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/"

PATHS = {
    "users": f"{RAW_PATH}users-500k-csv/",
    "products": f"{RAW_PATH}products-csv/",
    "events": f"{RAW_PATH}events-500k-json/",
    "sales": f"{RAW_PATH}sales-csv/",
    "users_historical": f"{RAW_PATH}users-historical/",
    "events_historical": f"{RAW_PATH}events-historical/",
    "sales_historical": f"{RAW_PATH}sales-historical/",
}

## Data Loading — spark.read

Each raw source is loaded into a Spark DataFrame using `spark.read`.

### Users


In [3]:
df_users = spark.read\
    .option("header", "true")\
    .option("inferSchema", "false")\
    .option("sep", "\t") \
    .csv(PATHS['users'])\
    .withColumn("_ingested_at", current_timestamp())\
    .withColumn("_source_file", col("_metadata.file_path"))

df_users.show(5)

+-----------------+--------------------------+--------------------+--------------------+--------------------+
|          user_id|user_first_touch_timestamp|               email|        _ingested_at|        _source_file|
+-----------------+--------------------------+--------------------+--------------------+--------------------+
|UA000000102357305|          1592182691348767|                NULL|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
|UA000000102357308|          1592183287634953|                NULL|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
|UA000000102357309|          1592183302736627|                NULL|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
|UA000000102357321|          1592184604178702|david23@orozco-pa...|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
|UA000000102357325|          1592185154063628|                NULL|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
+-----------------+--------------------------+--------------------+--------------------+--------------------+
only showi

### Product

In [4]:
df_products = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .csv(PATHS["products"]) \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))
    
df_products.show(5)

+--------+--------------------+------+--------------------+--------------------+
| item_id|                name| price|        _ingested_at|        _source_file|
+--------+--------------------+------+--------------------+--------------------+
|M_STAN_Q|Standard Queen Ma...|1045.0|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
|M_STAN_K|Standard King Mat...|1195.0|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
|M_STAN_T|Standard Twin Mat...| 595.0|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
|M_PREM_Q|Premium Queen Mat...|1795.0|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
|M_STAN_F|Standard Full Mat...| 945.0|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
+--------+--------------------+------+--------------------+--------------------+
only showing top 5 rows


### Events

In [5]:
df_events = spark.read \
    .json(PATHS["events"]) \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))
    
df_events.show()

+-------+------------------+----------+------------------------+----------------+--------------------+--------------------+--------------+--------------------------+-----------------+--------------------+--------------------+
| device|         ecommerce|event_name|event_previous_timestamp| event_timestamp|                 geo|               items|traffic_source|user_first_touch_timestamp|          user_id|        _ingested_at|        _source_file|
+-------+------------------+----------+------------------------+----------------+--------------------+--------------------+--------------+--------------------------+-----------------+--------------------+--------------------+
|  macOS|{NULL, NULL, NULL}|  warranty|        1593878899217692|1593878946592107|      {Montrose, MI}|                  []|        google|          1593878899217692|UA000000107379500|2026-06-07 15:33:...|dbfs:/Volumes/dat...|
|Windows|{NULL, NULL, NULL}|     press|        1593876662175340|1593877011756535|   {Northampton

###Sales

In [6]:
df_sales_csv = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("sep", "|") \
    .csv(PATHS["sales"]) \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))

df_sales_csv.show()

+--------+--------------------+----------------------+-------------------+-----------------------+------------+--------------------+--------------------+--------------------+
|order_id|               email|transactions_timestamp|total_item_quantity|purchase_revenue_in_usd|unique_items|               items|        _ingested_at|        _source_file|
+--------+--------------------+----------------------+-------------------+-----------------------+------------+--------------------+--------------------+--------------------+
|  298592|sandovalaustin@ho...|      1592629288475307|                  1|                  850.5|           1|[{'coupon': 'NEWB...|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|  299024|   msmith@monroe.com|      1592636869915092|                  2|                 1092.6|           2|[{'coupon': 'NEWB...|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|  300048|robertstimothy@ho...|      1592649862529478|                  1|                 1075.5|           1|[{'coupon': 'N

### Item lookup

In [7]:
df_item_lookup = spark.read\
    .parquet(f"{RAW_PATH}item-lookup/")\
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path"))


df_item_lookup.show()

+--------+--------------------+------+--------------------+--------------------+
| item_id|                name| price|        _ingested_at|        _source_file|
+--------+--------------------+------+--------------------+--------------------+
|M_PREM_Q|Premium Queen Mat...|1795.0|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|M_STAN_F|Standard Full Mat...| 945.0|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|M_PREM_F|Premium Full Matt...|1695.0|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|M_PREM_T|Premium Twin Matt...|1095.0|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|M_PREM_K|Premium King Matt...|1995.0|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|P_DOWN_S|Standard Down Pillow| 119.0|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|M_STAN_Q|Standard Queen Ma...|1045.0|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|M_STAN_K|Standard King Mat...|1195.0|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|M_STAN_T|Standard Twin Mat...| 595.0|2026-06-07 15:35:...|dbfs:/Volumes/dat...|
|P_FOAM_S|Standard Foam Pill

## Writing to Bronze — One JSON Delta Table

All entities are converted to JSON rows and written to a **single** Delta path:

`/Volumes/ecom_clickstream/bronze/delta/clickstream_json/`

Base sources use `overwrite`. Historical sources use `append`.


In [8]:
# Convert each source DataFrame to JSON rows and union them
bronze_base = (
    to_bronze_json_rows(df_users, "users")
    .unionByName(to_bronze_json_rows(df_products, "products"))
    .unionByName(to_bronze_json_rows(df_events, "events"))
    .unionByName(to_bronze_json_rows(df_sales_csv, "sales"))
    .unionByName(to_bronze_json_rows(df_item_lookup, "item_lookup"))
)

bronze_base.write.format("delta").mode("overwrite").save(BRONZE_JSON_PATH)
print(f"Bronze JSON table written: {bronze_base.count()} rows")
bronze_base.groupBy("entity_type").count().show()

Bronze JSON table written: 1010534 rows
+-----------+------+
|entity_type| count|
+-----------+------+
|      users|500000|
|   products|    12|
|     events|500000|
|      sales| 10510|
|item_lookup|    12|
+-----------+------+



In [9]:
# Append historical sources to the same JSON bronze table
df_users_historical = spark.read.parquet(PATHS["users_historical"])
hist_users = to_bronze_json_rows(
    df_users_historical
    .select([col(c).cast("string") for c in df_users_historical.columns])
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path")),
    "users",
)
hist_users.write.format("delta").mode("append").save(BRONZE_JSON_PATH)

df_events_historical = spark.read.parquet(PATHS["events_historical"])
hist_events = to_bronze_json_rows(
    df_events_historical
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path")),
    "events",
)
hist_events.write.format("delta").mode("append").save(BRONZE_JSON_PATH)

df_sales_historical = spark.read.parquet(PATHS["sales_historical"])
sales_hist_cols = [
    col(c).cast("string") if c != "items" else col(c)
    for c in df_sales_historical.columns
]
hist_sales = to_bronze_json_rows(
    df_sales_historical
    .select(sales_hist_cols)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path")),
    "sales",
)
hist_sales.write.format("delta").mode("append").save(BRONZE_JSON_PATH)

bronze_final = load_bronze(spark)
print(f"Bronze JSON final count: {bronze_final.count()}")
bronze_final.groupBy("entity_type").count().show()

Bronze JSON final count: 1758241
+-----------+------+
|entity_type| count|
+-----------+------+
|     events|985696|
|      users|751501|
|      sales| 21020|
|   products|    12|
|item_lookup|    12|
+-----------+------+



In [10]:
display(bronze_final)

,entity_type,payload,_ingested_at,_source_file
0,events,"{""device"":""iOS"",""ecommerce"":{},""event_name"":""mattresses"",""event_timestamp"":1593010211262141,""geo"":{""city"":""College Station"",""state"":""TX""},""items"":[],""traffic_source"":""instagram"",""user_first_touch_timestamp"":1593010211262141,""user_id"":""UA000000104785746""}",2026-06-07 15:35:25.926283,dbfs:/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/events-500k-json/part-00000-tid-309888144738233288-fab86c62-ff9f-4176-98c6-587f95ee9066-2365-1-c000.json
1,events,"{""device"":""macOS"",""ecommerce"":{},""event_name"":""reviews"",""event_previous_timestamp"":1593009103672468,""event_timestamp"":1593009399134652,""geo"":{""city"":""Chico"",""state"":""CA""},""items"":[],""traffic_source"":""google"",""user_first_touch_timestamp"":1593009103672468,""user_id"":""UA000000104779728""}",2026-06-07 15:35:25.926283,dbfs:/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/events-500k-json/part-00000-tid-309888144738233288-fab86c62-ff9f-4176-98c6-587f95ee9066-2365-1-c000.json
2,events,"{""device"":""macOS"",""ecommerce"":{},""event_name"":""add_item"",""event_previous_timestamp"":1593016822492747,""event_timestamp"":1593017131809883,""geo"":{""city"":""Tampa"",""state"":""FL""},""items"":[{""item_id"":""M_STAN_K"",""item_name"":""Standard King Mattress"",""item_revenue_in_usd"":1195.0,""price_in_usd"":1195.0,""quantity"":1}],""traffic_source"":""google"",""user_first_touch_timestamp"":1593016822492747,""user_id"":""UA000000104825536""}",2026-06-07 15:35:25.926283,dbfs:/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/events-500k-json/part-00000-tid-309888144738233288-fab86c62-ff9f-4176-98c6-587f95ee9066-2365-1-c000.json
3,events,"{""device"":""iOS"",""ecommerce"":{},""event_name"":""original"",""event_previous_timestamp"":1593016000518903,""event_timestamp"":1593019683930031,""geo"":{""city"":""Burien"",""state"":""WA""},""items"":[],""traffic_source"":""google"",""user_first_touch_timestamp"":1593016000518903,""user_id"":""UA000000104820428""}",2026-06-07 15:35:25.926283,dbfs:/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/events-500k-json/part-00000-tid-309888144738233288-fab86c62-ff9f-4176-98c6-587f95ee9066-2365-1-c000.json
4,events,"{""device"":""macOS"",""ecommerce"":{},""event_name"":""main"",""event_timestamp"":1593018824918051,""geo"":{""city"":""Minneapolis"",""state"":""MN""},""items"":[],""traffic_source"":""direct"",""user_first_touch_timestamp"":1593018824918051,""user_id"":""UA000000104838345""}",2026-06-07 15:35:25.926283,dbfs:/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/events-500k-json/part-00000-tid-309888144738233288-fab86c62-ff9f-4176-98c6-587f95ee9066-2365-1-c000.json
5,events,"{""device"":""Android"",""ecommerce"":{},""event_name"":""main"",""event_timestamp"":1593020932457100,""geo"":{""city"":""Los Angeles"",""state"":""CA""},""items"":[],""traffic_source"":""direct"",""user_first_touch_timestamp"":1593020932457100,""user_id"":""UA000000104851638""}",2026-06-07 15:35:25.926283,dbfs:/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/events-500k-json/part-00000-tid-309888144738233288-fab86c62-ff9f-4176-98c6-587f95ee9066-2365-1-c000.json
6,events,"{""device"":""Windows"",""ecommerce"":{},""event_name"":""mattresses"",""event_timestamp"":1593009578890512,""geo"":{""city"":""College Place"",""state"":""WA""},""items"":[],""traffic_source"":""facebook"",""user_first_touch_timestamp"":1593009578890512,""user_id"":""UA000000104782292""}",2026-06-07 15:35:25.926283,dbfs:/Volumes/databricks_simulated_e_commerce_clickstream_data/v01/raw/events-500k-json/part-00000-tid-309888144738233288-fab86c62-ff9f-4176-98c6-587f95ee9066-2365-1-c000.json
7,events,"{""device"":""Chrome OS"",""ecommerce"":{},""event_name"":""mattresses"",""event_timestamp"":1593018008048460,""geo"":{""city"":""Oakland"",""state"":""CA""},""items"":[],""traffic_source"":""google"",""user

_Skipped — legacy per-table bronze writes removed. All data is in `clickstream_json/`._

In [13]:
# Legacy cell intentionally left blank.

   → 12 rows ingested


### Events

In [14]:
df_events \
    .write.format("delta") \
    .mode("overwrite") \
    .save(f"{BRONZE_PATH}events/")


print(f"   → {df_events.count()} rows ingested")

   → 500000 rows ingested


### Sales

In [15]:
df_sales_csv \
    .write.format("delta") \
    .mode("overwrite") \
    .save(f"{BRONZE_PATH}sales/")

print(f"Sales CSV — {df_sales_csv.count()} rows ingested")

Sales CSV — 10510 rows ingested


### Item lookup


In [ ]:
df_item_lookup \
    .write.format("delta") \
    .mode("overwrite") \
    .save(f"{BRONZE_PATH}item_lookup/")

print(f"✅ Sales Historical — {df_item_lookup.count()} rows ingested")

✅ Sales Historical — 12 lignes ajoutées


## Historical Sources — Append to Bronze

Historical sources are **Parquet files** that contain data from before the main snapshot.
They are appended to the existing Bronze tables using `mode("append")` to build
a complete historical record in a single unified Delta table.
I had a problem appending the historical files due to different schema even thought they are the same column. so to make it easier 

### Users

**Why We Cast All Columns to String Before Appending**

When attempting to append `users_historical` to the Bronze `users` table, 
the following error was raised:
```
[DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 
'user_first_touch_timestamp' and 'user_first_touch_timestamp'. 
SQLSTATE: 22005
```

**Root Cause**

The same column existed in both sources but with **different types**:

| Source | Column | Type |
|---|---|---|
| `users-500k-csv` (base table) | `user_first_touch_timestamp` | `StringType` |
| `users-historical` (Parquet) | `user_first_touch_timestamp` | `LongType` |

**Solution**

Cast all columns of the historical DataFrame to `String` before appending,
so both sources share the same type and Delta Lake accepts the merge:
```python
.select([col(c).cast("string") for c in df_users_historical.columns])
```

PS same problem with the Sales csv and historical

In [17]:
df_users_historical = spark.read.parquet(f"{RAW_PATH}users-historical/")

In [ ]:
df_users_historical \
    .select([col(c).cast("string") for c in df_users_historical.columns])\
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(f"{BRONZE_PATH}users/")

print(f"Users historical ingested")

Users historical ajouté


### Events

In [19]:
df_events_historical = spark.read.parquet(f"{RAW_PATH}events-historical/")

In [20]:
df_events_historical \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(f"{BRONZE_PATH}events/")

print(f"✅ Sales Historical — {df_events_historical.count()} lignes ajoutées")

✅ Sales Historical — 485696 lignes ajoutées


### Sales

In [21]:
df_sales_historical = spark.read.parquet(f"{RAW_PATH}sales-historical/")

In [ ]:
df_sales_historical \
    .select([col(c).cast("string") for c in df_sales_historical.columns])\
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(f"{BRONZE_PATH}sales/")

print(f"✅ Sales Historical — {df_sales_historical.count()} historical rows ingested")

✅ Sales Historical — 10510 lignes ajoutées
